In [143]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.metrics import roc_auc_score
import json
from tqdm import tqdm
from catboost import CatBoostClassifier

In [4]:
file_path = './data/data_correct.json'
with open(file_path, 'r') as file:
        data = json.load(file)    

In [5]:
df = pd.DataFrame(data)

In [6]:
df['domain'].value_counts()

domain
human      34713
Llama2     17214
chatgpt     4154
Llama3      1966
GPT3.5      1882
Qwen        1868
Name: count, dtype: int64

In [7]:
data[11]

{'text': 'The paper effectively addresses an important problem in the field of entity recognition by proposing a new model based on multigraphs. This approach is novel and has the potential to overcome the limitations of existing models that utilize simple graphs. The experiments conducted on standard datasets provide evidence of the superior performance of the proposed model compared to previous models. The results indicate that the multigraph-based model is capable of accurately predicting overlapping entities, which is a challenging task. The analysis presented in the paper on the differences between the proposed model and previous models sheds light on the underlying mechanisms and highlights the strengths of the multigraph-based approach. This contributes to a better understanding of the problem and provides valuable insights for future research. The claim that this is the first structured prediction model utilizing multigraphs for predicting overlapping structures is significant,

In [8]:
ppl_llama3 = np.load('./data/ppl_llama3.npy')
ppl_llama2 = np.load('./data/ppl_llama2.npy')
ppl_qwen = np.load('./data/ppl_qwen.npy')

In [9]:
ppl_llama3

array([27.469696, 13.407839, 22.490164, ..., 15.157976,  4.548979,
        9.542872], dtype=float32)

In [10]:
ppl_llama2

array([ 5812.8022,  3171.1804,  3527.042 , ..., 14197.993 ,  7160.3774,
        5744.6646], dtype=float32)

In [11]:
ppl_qwen

array([24.274326 , 13.989251 , 23.692776 , ..., 21.28179  ,  7.3349366,
        8.859068 ], dtype=float32)

In [15]:
data = pd.DataFrame(data)

In [16]:
data.groupby('label').domain.value_counts()

label         domain 
human_text    human      34713
machine_text  Llama2     17214
              chatgpt     4154
              Llama3      1966
              GPT3.5      1882
              Qwen        1868
Name: count, dtype: int64

In [17]:
data['ppl_llama3'] = ppl_llama3
data['ppl_llama2'] = ppl_llama2
data['ppl_qwen'] = ppl_qwen

In [18]:
data

,text,label,domain,ppl_llama3,ppl_llama2,ppl_qwen
0,- Strengths:\n* Outperforms ALIGN in supervise...,human_text,human,27.469696,5812.802246,24.274326
1,the mention or also includes other related ent...,machine_text,chatgpt,13.407839,3171.180420,13.989251
2,This paper addresses the problem of disambigua...,human_text,human,22.490164,3527.041992,23.692776
3,For entities: their profiles consist of neighb...,machine_text,chatgpt,9.320074,3041.216553,12.446774
4,"- Strengths:\nGood ideas, simple neural learni...",human_text,human,33.169468,3590.321533,34.518867
...,...,...,...,...,...,...
61792,Bobby Douglas also suggested an American woman...,human_text,human,18.652077,4219.416504,19.506374
61793,"Ursula von der Leyen, a close ally of Chancell...",human_text,human,11.092178,6800.665039,12.950030
61794,The hosts' display was uncertain and their opp...,machine_text,Qwen,15.157976,14197.993164,21.281790
61795,Birkhoff-Smale theorem says that transverse ho...,machine_text,Llama3,4.548979,7160.377441,7.334937


In [21]:
data['target'] = data['domain'].apply(lambda x: 1 if x in ['Qwen', 'Llama2', 'Llama3'] else 0)

In [22]:
data

,text,label,domain,ppl_llama3,ppl_llama2,ppl_qwen,target
0,- Strengths:\n* Outperforms ALIGN in supervise...,human_text,human,27.469696,5812.802246,24.274326,0
1,the mention or also includes other related ent...,machine_text,chatgpt,13.407839,3171.180420,13.989251,0
2,This paper addresses the problem of disambigua...,human_text,human,22.490164,3527.041992,23.692776,0
3,For entities: their profiles consist of neighb...,machine_text,chatgpt,9.320074,3041.216553,12.446774,0
4,"- Strengths:\nGood ideas, simple neural learni...",human_text,human,33.169468,3590.321533,34.518867,0
...,...,...,...,...,...,...,...
61792,Bobby Douglas also suggested an American woman...,human_text,human,18.652077,4219.416504,19.506374,0
61793,"Ursula von der Leyen, a close ally of Chancell...",human_text,human,11.092178,6800.665039,12.950030,0
61794,The hosts' display was uncertain and their opp...,machine_text,Qwen,15.157976,14197.993164,21.281790,1
61795,Birkhoff-Smale theorem says that transverse ho...,machine_text,Llama3,4.548979,7160.377441,7.334937,1


In [24]:
data['is_llama2'] = (data['domain'].apply(lambda x: 'Llama2' == x))

In [23]:
data['is_llama3'] = (data['domain'].apply(lambda x: 'Llama3' == x))

In [25]:
data['is_qwen'] = (data['domain'].apply(lambda x: 'Qwen' == x))

In [26]:
data

,text,label,domain,ppl_llama3,ppl_llama2,ppl_qwen,target,is_llama3,is_llama2,is_qwen
0,- Strengths:\n* Outperforms ALIGN in supervise...,human_text,human,27.469696,5812.802246,24.274326,0,False,False,False
1,the mention or also includes other related ent...,machine_text,chatgpt,13.407839,3171.180420,13.989251,0,False,False,False
2,This paper addresses the problem of disambigua...,human_text,human,22.490164,3527.041992,23.692776,0,False,False,False
3,For entities: their profiles consist of neighb...,machine_text,chatgpt,9.320074,3041.216553,12.446774,0,False,False,False
4,"- Strengths:\nGood ideas, simple neural learni...",human_text,human,33.169468,3590.321533,34.518867,0,False,False,False
...,...,...,...,...,...,...,...,...,...,...
61792,Bobby Douglas also suggested an American woman...,human_text,human,18.652077,4219.416504,19.506374,0,False,False,False
61793,"Ursula von der Leyen, a close ally of Chancell...",human_text,human,11.092178,6800.665039,12.950030,0,False,False,False
61794,The hosts' display was uncertain and their opp...,machine_text,Qwen,15.157976,14197.993164,21.281790,1,False,False,True
61795,Birkhoff-Smale theorem says that transverse ho...,machine_text,Llama3,4.548979,7160.377441,7.334937,1,True,False,False


In [75]:
best_i_llama3 = 0
best_score_llama3 = 0
best_i_llama2 = 0
best_score_llama2 = 0
best_i_qwen = 0
best_score_qwen = 0

for i in tqdm(data.index):
    pred = data['ppl_llama3'] < data['ppl_llama3'][i]
    # score = f1_score(data['is_gpt'], pred)
    score = (data['is_llama3'] == pred).sum()
    if score > best_score_llama3:
        best_i_llama3 = data['ppl_llama3'][i]
        best_score_llama3 = score
    pred = data['ppl_llama2'] < data['ppl_llama2'][i]
    # score = f1_score(data['is_llama'], pred)
    score = (data['is_llama2'] == pred).sum()
    if score > best_score_llama2:
        best_i_llama2 = data['ppl_llama2'][i]
        best_score_llama2 = score
    pred = data['ppl_qwen'] < data['ppl_qwen'][i]
    # score = f1_score(data['is_qwen'], pred)
    score = (data['is_qwen'] == pred).sum()
    if score > best_score_qwen:
        best_i_qwen = data['ppl_qwen'][i]
        best_score_qwen = score

100%|██████████████████████████████████████████████████████████████████████████| 61700/61700 [00:43<00:00, 1420.25it/s]


In [179]:
print(f"Максимальное значение perplexity Llama 2 {max(ppl_llama2)},\nГраничное значение для Llama2 {best_i_llama2},\nЛучший скор для Llama2 {best_score_llama2}")

Максимальное значение perplexity Llama 2 54845.18359375,
Граничное значение для Llama2 1786.2454833984375,
Лучший скор для Llama2 44726


In [180]:
print(f"Максимальное значение perplexity Llama 3 {max(ppl_llama3)},\nГраничное значение для Llama3 {best_i_llama3},\nЛучший скор для Llama3 {best_score_llama3}")

Максимальное значение perplexity Llama 3 137674.734375,
Граничное значение для Llama3 1.1858340501785278,
Лучший скор для Llama3 59734


In [182]:
print(f"Максимальное значение perplexity Qwen {max(ppl_qwen)},\nГраничное значение для Qwen {best_i_qwen},\nЛучший скор для Qwen {best_score_qwen}")

Максимальное значение perplexity Qwen 202628.5,
Граничное значение для Qwen 1.1632660627365112,
Лучший скор для Qwen 59832


In [71]:
data[data['ppl_qwen'].apply(lambda x: math.isnan(x))]

,text,label,domain,ppl_llama3,ppl_llama2,ppl_qwen,target,is_llama3,is_llama2,is_qwen


In [113]:
data[(data['ppl_llama3'] > best_i_llama3) & (data['ppl_llama2'] > best_i_llama2) & (data['ppl_qwen'] > best_i_qwen)]

,text,label,domain,ppl_llama3,ppl_llama2,ppl_qwen,target,is_llama3,is_llama2,is_qwen
0,- Strengths:\n* Outperforms ALIGN in supervise...,human_text,human,27.469696,5812.802246,24.274326,0,False,False,False
1,the mention or also includes other related ent...,machine_text,chatgpt,13.407839,3171.180420,13.989251,0,False,False,False
2,This paper addresses the problem of disambigua...,human_text,human,22.490164,3527.041992,23.692776,0,False,False,False
3,For entities: their profiles consist of neighb...,machine_text,chatgpt,9.320074,3041.216553,12.446774,0,False,False,False
4,"- Strengths:\nGood ideas, simple neural learni...",human_text,human,33.169468,3590.321533,34.518867,0,False,False,False
...,...,...,...,...,...,...,...,...,...,...
61792,Bobby Douglas also suggested an American woman...,human_text,human,18.652077,4219.416504,19.506374,0,False,False,False
61793,"Ursula von der Leyen, a close ally of Chancell...",human_text,human,11.092178,6800.665039,12.950030,0,False,False,False
61794,The hosts' display was uncertain and their opp...,machine_text,Qwen,15.157976,14197.993164,21.281790,1,False,False,True
61795,Birkhoff-Smale theorem says that transverse ho...,machine_text,Llama3,4.548979,7160.377441,7.334937,1,True,False,False


In [114]:
data.target.value_counts()

target
0    40749
1    20951
Name: count, dtype: int64

In [183]:
X = data[['ppl_llama3', 'ppl_llama2', 'ppl_qwen']]
y = data.target

In [184]:
X_train, X_test, y_train, y_test = train_test_split(data[['ppl_llama3', 'ppl_llama2', 'ppl_qwen']], data.target, test_size=0.25, stratify=data.target)

In [185]:
from sklearn.preprocessing import MinMaxScaler

mms = MinMaxScaler().set_output(transform='pandas')
mms.fit(X_train)

X_train = mms.transform(X_train)
X_test = mms.transform(X_test)

In [186]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

LogisticRegression()

In [187]:
pred_train = lr.predict(X_train)
pred_test = lr.predict(X_test)

In [188]:
print(classification_report(y_train, pred_train))
print(classification_report(y_test, pred_test))

              precision    recall  f1-score   support

           0       0.66      1.00      0.80     30562
           1       1.00      0.00      0.00     15713

    accuracy                           0.66     46275
   macro avg       0.83      0.50      0.40     46275
weighted avg       0.78      0.66      0.53     46275

              precision    recall  f1-score   support

           0       0.66      1.00      0.80     10187
           1       1.00      0.00      0.00      5238

    accuracy                           0.66     15425
   macro avg       0.83      0.50      0.40     15425
weighted avg       0.78      0.66      0.53     15425



In [189]:
pred_train = lr.predict_proba(X_train)
pred_test = lr.predict_proba(X_test)

In [191]:
from sklearn.metrics import roc_auc_score

print(f"ROC-AUC на трейне: {roc_auc_score(y_train, pred_train[:,1])}")
print(f"ROC-AUC на тесте: {roc_auc_score(y_test, pred_test[:,1])}")

ROC-AUC на трейне: 0.576690205232425
ROC-AUC на тесте: 0.5786115973412498


In [192]:
cb = CatBoostClassifier(verbose=False)

In [193]:
cb.fit(X_train, y_train)

In [194]:
pred_train = cb.predict(X_train)
pred_test = cb.predict(X_test)

In [195]:
print(classification_report(y_train, pred_train))
print(classification_report(y_test, pred_test))

              precision    recall  f1-score   support

           0       0.85      0.90      0.87     30562
           1       0.78      0.68      0.73     15713

    accuracy                           0.83     46275
   macro avg       0.81      0.79      0.80     46275
weighted avg       0.82      0.83      0.82     46275

              precision    recall  f1-score   support

           0       0.83      0.89      0.86     10187
           1       0.75      0.65      0.70      5238

    accuracy                           0.81     15425
   macro avg       0.79      0.77      0.78     15425
weighted avg       0.81      0.81      0.81     15425



In [196]:
pred_train = cb.predict_proba(X_train)
pred_test = cb.predict_proba(X_test)

In [198]:
print(f"ROC-AUC на трейне: {roc_auc_score(y_train, pred_train[:,1])}")
print(f"ROC-AUC на тесте: {roc_auc_score(y_test, pred_test[:,1])}")

ROC-AUC на трейне: 0.8839402074845144
ROC-AUC на тесте: 0.8583649743683909


In [199]:
import optuna
from sklearn.model_selection import cross_val_score
def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 20),
        'verbose': False,
        'early_stopping_rounds': 50,
    }
    
    # Используем кросс-валидацию
    model = CatBoostClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=3, scoring='roc_auc')
    return np.mean(scores)
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print("Лучшие параметры:", study.best_params)
print("Лучшее значение AUC-ROC:", study.best_value)

[I 2025-05-14 22:59:44,040] A new study created in memory with name: no-name-775822bc-e1e1-4351-8ed2-df367498992c
[I 2025-05-14 22:59:52,842] Trial 0 finished with value: 0.8357894713300169 and parameters: {'iterations': 217, 'depth': 8, 'learning_rate': 0.04124775538983445, 'l2_leaf_reg': 1.3678203065649002, 'random_strength': 7.879382672028832, 'bagging_temperature': 0.25144400561527225, 'min_data_in_leaf': 3}. Best is trial 0 with value: 0.8357894713300169.
[I 2025-05-14 23:00:19,896] Trial 1 finished with value: 0.8666827625316417 and parameters: {'iterations': 644, 'depth': 8, 'learning_rate': 0.18524186971325907, 'l2_leaf_reg': 7.5555533434353315, 'random_strength': 6.891354866698466, 'bagging_temperature': 0.4960155949684345, 'min_data_in_leaf': 13}. Best is trial 1 with value: 0.8666827625316417.
[I 2025-05-14 23:00:52,223] Trial 2 finished with value: 0.8673631052914278 and parameters: {'iterations': 423, 'depth': 9, 'learning_rate': 0.19370022565713638, 'l2_leaf_reg': 5.68826

Лучшие параметры: {'iterations': 817, 'depth': 10, 'learning_rate': 0.2677276657238807, 'l2_leaf_reg': 6.122800933328357, 'random_strength': 8.728944195423962, 'bagging_temperature': 0.015598614008097866, 'min_data_in_leaf': 7}
Лучшее значение AUC-ROC: 0.877340754839922


In [ ]:
best_model = CatBoostClassifier(
    **study.best_params, verbose=100
)

best_model.fit(X_train, y_train)

0:	learn: 0.5986220	total: 56.6ms	remaining: 46.2s
100:	learn: 0.4047965	total: 3.97s	remaining: 28.2s
200:	learn: 0.3574028	total: 8.07s	remaining: 24.7s
300:	learn: 0.3242399	total: 12.2s	remaining: 20.9s
400:	learn: 0.2991669	total: 16.4s	remaining: 17s
500:	learn: 0.2756278	total: 20.6s	remaining: 13s
600:	learn: 0.2560708	total: 24.8s	remaining: 8.9s
700:	learn: 0.2384275	total: 29s	remaining: 4.8s


In [ ]:
pred_train = best_model.predict(X_train)
pred_test = best_model.predict(X_test)

In [ ]:
print(classification_report(y_train, pred_train))
print(classification_report(y_test, pred_test))

In [ ]:
pred_train = best_model.predict_proba(X_train)
pred_test = best_model.predict_proba(X_test)

In [ ]:
print(f"ROC-AUC на трейне: {roc_auc_score(y_train, pred_train[:,1])}")
print(f"ROC-AUC на тесте: {roc_auc_score(y_test, pred_test[:,1])}")